In [ ]:
from pathlib import Path
import sys
sys.path.append(str(Path.cwd().parent))

from src.data_loader import load_data
import importlib
importlib.reload(src.preprocessing)
importlib.reload(src.model)
import src.preprocessing
import src.model

Load data and preprocessing steps(split data, remove invalid values, handle outlier values, impute missing values):

In [66]:
PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "cs-training.csv"

df = load_data(DATA_PATH)
train_df, test_df = src.preprocessing.split_data(df, test_size=0.2, random_state=67)

train_df = src.preprocessing.remove_invalid_ages(train_df)
test_df = src.preprocessing.remove_invalid_ages(test_df)

train_df = src.preprocessing.handle_sentinel_values(train_df)
test_df = src.preprocessing.handle_sentinel_values(test_df)

train_df, test_df = src.preprocessing.handle_debt_ratio_outlier(train_df,test_df)

train_df, test_df = src.preprocessing.handle_revolving_utilization_outlier(train_df, test_df)

train_df, test_df = src.preprocessing.impute_missing(train_df=train_df, 
                                       test_df=test_df, 
                                       col="MonthlyIncome")
train_df, test_df = src.preprocessing.impute_missing(train_df=train_df, 
                                       test_df=test_df, 
                                       col="NumberOfDependents")


In [67]:
BASELINE_FEATURES = [
    "RevolvingUtilizationOfUnsecuredLines",
    "age",
    "NumberOfTime30-59DaysPastDueNotWorse",
    "DebtRatio",
    "MonthlyIncome",
    "NumberOfOpenCreditLinesAndLoans",
    "NumberOfTimes90DaysLate",
    "NumberRealEstateLoansOrLines",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfDependents"
]  

In [68]:
X_train, y_train = src.model.prepare_features(train_df, "SeriousDlqin2yrs", BASELINE_FEATURES)
X_test, y_test = src.model.prepare_features(test_df, "SeriousDlqin2yrs", BASELINE_FEATURES)

baseline_model = src.model.fit_logistic_model(train_df,"SeriousDlqin2yrs",BASELINE_FEATURES)

train_prob = src.model.predict_probabilities(baseline_model, train_df,BASELINE_FEATURES)
test_prob = src.model.predict_probabilities(baseline_model, test_df,BASELINE_FEATURES)
# print(train_prob)
# print(test_prob)

In [69]:
test_metrics = src.model.evaluate_model(y_test, test_prob)
test_metrics

{'ROC-AUC': 0.8519634610646127, 'KS': None}

In [57]:
# MonthlyIncome has a large range and doesn't scale well

train_df[BASELINE_FEATURES].describe().T
print(train_df["MonthlyIncome"].nlargest(20))
print(train_df["MonthlyIncome"].describe(percentiles=[.90, .95, .99, .995, .999]))

      
73764     3008750.0
111366    1560100.0
50641     1072500.0
122544     835040.0
123292     730483.0
93565      702500.0
35974      582369.0
137427     562466.0
114763     440000.0
106341     304000.0
43345      287662.0
23700      261666.0
34771      250000.0
46680      237490.0
17598      235000.0
67679      234600.0
267        208333.0
4044       203500.0
77169      184903.0
90029      173000.0
Name: MonthlyIncome, dtype: float64
count    1.197920e+05
mean     6.419882e+03
std      1.260370e+04
min      0.000000e+00
90%      1.080000e+04
95%      1.354235e+04
99%      2.320450e+04
99.5%    3.124045e+04
99.9%    7.200000e+04
max      3.008750e+06
Name: MonthlyIncome, dtype: float64


Data quality issue with debt ratio. 111 values in the training set and 17 valus in the test set remained with impossible high debt ratio values. Went and clipped values above 10 in order to fix issue.

In [ ]:
train_df.loc[
    train_df["MonthlyIncome"] > 100000,
    [
        "MonthlyIncome",
        "age",
        "DebtRatio",
        "RevolvingUtilizationOfUnsecuredLines",
        "SeriousDlqin2yrs"
    ]
].sort_values(
    "MonthlyIncome",
    ascending=False
).head(20)

train_df.loc[
    train_df["DebtRatio"] > 10,
    [
        "DebtRatio",
        "MonthlyIncome",
        "age",
        "RevolvingUtilizationOfUnsecuredLines",
        "SeriousDlqin2yrs"
    ]
].sort_values(
    "DebtRatio",
    ascending=False
).head(30)

train_df.loc[
    train_df["DebtRatio"] > 10,
    [
        "DebtRatio",
        "MonthlyIncome",
        "age",
        "RevolvingUtilizationOfUnsecuredLines",
        "SeriousDlqin2yrs"
    ]
].sort_values(
    "DebtRatio",
    ascending=False
).head(30)

(test_df["DebtRatio"] > 10).sum()

np.int64(0)

Creates implied debt

In [ ]:
df["ImpliedDebt"] = (df["DebtRatio"] * df["MonthlyIncome"])
extreme_debt = train_df[train_df["DebtRatio"] > 10].copy()

extreme_debt["ImpliedDebt"] = (
    extreme_debt["DebtRatio"] * extreme_debt["MonthlyIncome"]
)